In [2]:
import pandas as pd
import re
import math
from collections import Counter

df = pd.read_csv('bbc-text (1).csv')

# Show the first 2 rows and dataset info
print("=== First 2 rows of the dataset ===")
display(df.head(2))
print("\nDataset Shape:", df.shape)

=== First 2 rows of the dataset ===


,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...



Dataset Shape: (2225, 2)


In [3]:
# Define a simple stop words list
stop_words = set(['the', 'a', 'an', 'and', 'in', 'on', 'at', 'for', 'to', 'of', 'is', 'are', 'it', 'this', 'that', 'with', 'as', 'was', 'by', 'from', 'be', 'which', 'or', 'not', 'were', 'have', 'has', 'had', 'they', 'you', 'he', 'she', 'we', 'his', 'her', 'their', 'our'])

def preprocess_text(text):
    # 1. Convert to lowercase
    text = str(text).lower()
    # 2. Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # 3. Split into words (tokens)
    tokens = text.split()
    # 4. Remove stop words and short words
    filtered_tokens = [w for w in tokens if w not in stop_words and len(w) > 1]
    return filtered_tokens

# Apply preprocessing to all documents in the dataframe
df['tokens'] = df['text'].apply(preprocess_text)

print("=== Preprocessing completed ===")
display(df[['text', 'tokens']].head(2))

=== Preprocessing completed ===


,text,tokens
0,tv future in the hands of viewers with home th...,"[tv, future, hands, viewers, home, theatre, sy..."
1,worldcom boss left books alone former worldc...,"[worldcom, boss, left, books, alone, former, w..."


In [4]:
# Build vocabulary from all tokens in the dataset
all_tokens = [word for doc_tokens in df['tokens'] for word in doc_tokens]
vocab = sorted(list(set(all_tokens)))

# Create a mapping from word to index
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}

print(f"Total vocabulary size (V): {len(vocab)}")

# Function to convert a document's tokens into a BoW vector
def get_bow_vector(tokens, vocab_dict):
    vector = [0] * len(vocab_dict)
    counts = Counter(tokens)
    for word, count in counts.items():
        if word in vocab_dict:
            vector[vocab_dict[word]] = count
    return vector

# Generate BoW vectors for all documents
bow_vectors = [get_bow_vector(tokens, word_to_idx) for tokens in df['tokens']]
print("BoW vectors created successfully!")

Total vocabulary size (V): 30279
BoW vectors created successfully!


In [5]:
# Number of total documents (N)
N = len(df)

# Calculate Inverse Document Frequency (IDF) for each word in vocabulary
idf = {}
for word in vocab:
    # Count how many documents contain the word
    doc_count = sum(1 for tokens in df['tokens'] if word in tokens)
    # Using smoothed IDF formula: ln((1 + N) / (1 + doc_count)) + 1
    idf[word] = math.log((1 + N) / (1 + doc_count)) + 1

# Function to calculate TF-IDF vector for a document
def get_tfidf_vector(tokens, vocab_dict, idf_dict):
    counts = Counter(tokens)
    doc_len = len(tokens) if len(tokens) > 0 else 1
    vector = []
    for word in vocab_dict:
        # Term Frequency (TF): normalized count
        tf = counts[word] / doc_len
        # TF-IDF = TF * IDF
        tf_idf_val = tf * idf_dict[word]
        vector.append(tf_idf_val)
    return vector

# Generate TF-IDF vectors for all documents (Note: this might take a minute for the full dataset)
tfidf_vectors = [get_tfidf_vector(tokens, word_to_idx, idf) for tokens in df['tokens']]
print("TF-IDF vectors created successfully!")

TF-IDF vectors created successfully!


In [6]:
import numpy as np

# Add category column back and convert tfidf_vectors to a dataframe for analysis
tfidf_df = pd.DataFrame(tfidf_vectors, columns=vocab)
tfidf_df['category'] = df['category'].values

# Find top 10 words with highest average TF-IDF score for a specific category (e.g., 'tech')
target_category = 'tech'
cat_subset = tfidf_df[tfidf_df['category'] == target_category].drop(columns=['category'])

mean_scores = cat_subset.mean(axis=0)
top_10_words = mean_scores.nlargest(10)

print(f"=== Top 10 words for category '{target_category}' based on TF-IDF ===")
print(top_10_words)

=== Top 10 words for category 'tech' based on TF-IDF ===
people        0.015085
said          0.014754
will          0.013214
users         0.012994
software      0.012873
mobile        0.012631
more          0.011708
technology    0.011560
can           0.010858
microsoft     0.010601
dtype: float64
